# 13｜深層神經網路訓練_最佳化與正則化

在同一個狹長損失面比較一般梯度下降與 Momentum。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/13_深層神經網路訓練_最佳化與正則化.md`，目前 53 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

六種最佳化器、排程、Dropout/MC、Max-norm、早停與 CSV 記錄。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 13-03 | 4、5、6 | 課堂小實驗 |
| 13-04 | 3、5 | 執行結果圖 |
| 13-05 | 4、5、6、7、8、9、10、11、12、13、14、16、17、18、19、20、33、34、53 | 1. 六種更新規則與二維路徑 |
| 13-06 | 21、22、23、24、25、26、27、28、29、30、53 | 2. 固定、指數、餘弦、重啟、暖身與 1cycle |
| 13-07 | 32、33、34、35、36、37、38、39、40、41、42、43、44、45、46、47、48、49、53 | 3. Dropout、MC 預測、Max-norm 與早停快照 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 13-01｜投影片 13：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 13-02｜投影片 13：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

原函式記錄更新前損失；表格另以最後參數重算 final_loss，避免把第 79 步曲線與第 80 步參數混在一起。速度 buffer 不含學習率，與投影片含學習率的 v 記號不同。

In [ ]:
# 程式 13-03｜投影片 13：4、5、6 頁
A = np.diag([1.0, 30.0])
def optimize(momentum=0.0, eta=0.035, steps=80):
    theta = np.array([4.0, 1.5]); velocity = np.zeros(2); losses = []
    for _ in range(steps):
        losses.append(0.5 * theta @ A @ theta)
        grad = A @ theta
        velocity = momentum * velocity + grad
        theta -= eta * velocity
    return np.array(losses), theta

plain_loss, plain_theta = optimize(momentum=0.0)
mom_loss, mom_theta = optimize(momentum=0.8)
pd.DataFrame({'method': ['GD', 'Momentum'], 'final_loss': [0.5*plain_theta@A@plain_theta, 0.5*mom_theta@A@mom_theta],
              'theta_0': [plain_theta[0], mom_theta[0]], 'theta_1': [plain_theta[1], mom_theta[1]]})

## 執行結果圖

In [ ]:
# 程式 13-04｜投影片 13：3、5 頁
plt.semilogy(plain_loss, label='gradient descent')
plt.semilogy(mom_loss, label='momentum=0.8')
plt.xlabel('update'); plt.ylabel('quadratic loss'); plt.title('Momentum crosses the shallow direction faster')
plt.legend()
savefig('13_optimizer_regularization_demo')
report('optimizer_regularization', {'gd_final_loss': plain_loss[-1], 'momentum_final_loss': mom_loss[-1]})

## 1. 六種更新規則與二維路徑

所有方法從同一點出發；全域學習率依演算法固定列出，這是機制比較而非調參後的排名。Nesterov 使用投影片含學習率的速度慣例。

In [ ]:
# 程式 13-05｜投影片 13：4、5、6、7、8、9、10、11、12、13、14、16、17、18、19、20、33、34、53 頁
def path13(name,eta,steps=80):
    theta=np.array([4.,1.5]);v=np.zeros(2);s=np.zeros(2);m=np.zeros(2);path=[theta.copy()]
    for t13 in range(1,steps+1):
        g=A@theta
        if name=='GD':theta-=eta*g
        elif name=='Momentum':v=.8*v-eta*g;theta+=v
        elif name=='Nesterov':v=.8*v-eta*(A@(theta+.8*v));theta+=v
        elif name=='AdaGrad':s+=g*g;theta-=eta*g/(np.sqrt(s)+1e-8)
        elif name=='RMSProp':s=.9*s+.1*g*g;theta-=eta*g/(np.sqrt(s)+1e-8)
        elif name=='Adam':
            m=.9*m+.1*g;s=.999*s+.001*g*g
            theta-=eta*(m/(1-.9**t13))/(np.sqrt(s/(1-.999**t13))+1e-8)
        path.append(theta.copy())
    return np.array(path)
rates13={'GD':.035,'Momentum':.035,'Nesterov':.025,'AdaGrad':.5,'RMSProp':.1,'Adam':.1}
u13,v13=np.meshgrid(np.linspace(-2,4.5,160),np.linspace(-2,2,160));fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].contour(u13,v13,.5*(u13*u13+30*v13*v13),levels=[.1,1,5,15,30,60])
for name,rate in rates13.items():
    path=path13(name,rate);loss=.5*np.einsum('bi,ij,bj->b',path,A,path)
    axes[0].plot(*path.T,label=name);axes[1].semilogy(loss,label=name)
axes[0].set_title('Parameter paths');axes[1].set(xlabel='updates',ylabel='quadratic loss');axes[1].legend();savefig('c13_optimizer_paths')
v_hand13=0.;theta_hand13=1.
for _ in range(2):v_hand13=.9*v_hand13-.1*2;theta_hand13+=v_hand13
assert np.isclose(theta_hand13,.42)
m13=.1*2;s13=.001*4;assert np.allclose([m13/.1,s13/.001],[2,4])
# 同一狀態比較耦合 L2 與解耦 AdamW
w13=np.array([1.,3.]);g13=np.array([.2,2.]);lam13=.1;eta13=.01
coupled13=w13-eta13*(g13+lam13*w13)/(np.abs(g13+lam13*w13)+1e-8)
decoupled13=(1-eta13*lam13)*w13-eta13*g13/(np.abs(g13)+1e-8)
assert not np.allclose(coupled13,decoupled13)
print('L2 in Adam / AdamW first update:',coupled13,decoupled13)
# AdaMax 的最大絕對梯度狀態；二階步可直接解二次函數
print('AdaMax state:',np.maximum(.999*np.array([.5,1.]),np.abs(g13)))
print('Newton reaches quadratic optimum:',np.array([4.,1.5])-np.linalg.solve(A,A@np.array([4.,1.5])))

## 2. 固定、指數、餘弦、重啟、暖身與 1cycle

以 update 為時間軸，排程只改全域學習率。Plateau 依每輪驗證損失更新；batch 型排程每批一次，不能與 epoch 計數混用。

In [ ]:
# 程式 13-06｜投影片 13：21、22、23、24、25、26、27、28、29、30、53 頁
steps13=np.arange(100);eta_max13=.05
schedules13={'constant':np.full(100,eta_max13),'exponential':eta_max13*.96**steps13,
 'cosine':.001+(.05-.001)*(1+np.cos(np.pi*steps13/99))/2,
 'restarts':.001+(.05-.001)*(1+np.cos(np.pi*(steps13%25)/25))/2,
 'warmup+cosine':np.r_[np.linspace(.001,.05,10),.001+.049*(1+np.cos(np.pi*np.arange(90)/89))/2],
 '1cycle':np.r_[np.linspace(.002,.05,30),np.linspace(.05,.0001,70)]}
fig,axes=plt.subplots(1,2,figsize=(12,4))
for name,lrs in schedules13.items():
    theta=np.array([4.,1.5]);losses=[]
    for lr in lrs:theta-=lr*(A@theta);losses.append(.5*theta@A@theta)
    axes[0].plot(lrs,label=name);axes[1].semilogy(losses,label=name)
axes[0].set(xlabel='optimizer update',ylabel='learning rate');axes[0].legend(fontsize=8)
axes[1].set(xlabel='optimizer update',ylabel='quadratic loss');savefig('c13_lr_schedules')
plt.figure()
for rate in [.005,.04,.08]:
    path=path13('GD',rate,40);plt.semilogy(.5*np.einsum('bi,ij,bj->b',path,A,path),label=f'eta={rate}')
plt.legend();plt.xlabel('update');plt.ylabel('loss');savefig('c13_fixed_lr')
validation13=[1.,.8,.7,.71,.72,.69,.70,.71];best13=np.inf;bad13=0;lr13=.1;history13=[]
for value13 in validation13:
    if value13<best13:best13=value13;bad13=0
    else:bad13+=1
    if bad13>=2:lr13*=.5;bad13=0
    history13.append(lr13)
print('Plateau LR after each validation:',history13)

## 3. Dropout、MC 預測、Max-norm 與早停快照

用玩具網路驗算 inverted Dropout 的期望值與推論差異。MC 的方差只代表此模型與遮罩下的變動，不是校準過的總不確定性。早停用驗證選快照；記錄存在 CSV 可重新讀回。

In [ ]:
# 程式 13-07｜投影片 13：32、33、34、35、36、37、38、39、40、41、42、43、44、45、46、47、48、49、53 頁
import copy
rng13=np.random.default_rng(SEED);a13=np.array([1.,2.,3.,4.]);keep13=.5
masks13=rng13.binomial(1,keep13,size=(4000,4));dropped13=a13*masks13/keep13
assert np.allclose(dropped13.mean(0),a13,atol=.15)
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].imshow(masks13[:12],cmap='binary',aspect='auto');axes[0].set(title='Training masks',xlabel='hidden unit',ylabel='forward pass')
mc13=dropped13@np.array([.1,.2,-.1,.3]);axes[1].hist(mc13,bins=20);axes[1].axvline(a13@np.array([.1,.2,-.1,.3]),c='r',label='eval, no dropout');axes[1].legend();savefig('c13_dropout_mc')
weights13=np.array([[3.,4.],[0.,10.]])
projected13=weights13*np.minimum(1,3/np.linalg.norm(weights13,axis=1,keepdims=True));assert np.all(np.linalg.norm(projected13,axis=1)<=3+1e-8)
# 合成 noisy regression：訓練／驗證分開；最佳快照非最後一步參數
train_x13=rng13.normal(size=(30,20));train_y13=train_x13[:,0]+rng13.normal(0,.7,30)
valid_x13=rng13.normal(size=(100,20));valid_y13=valid_x13[:,0]+rng13.normal(0,.7,100)
w13=np.zeros(20);best_loss13=np.inf;best_w13=None;records13=[];bad13=0
for epoch13 in range(500):
    g13=train_x13.T@(train_x13@w13-train_y13)/len(train_y13);w13-=.05*g13
    vl13=np.mean((valid_x13@w13-valid_y13)**2);tr13=np.mean((train_x13@w13-train_y13)**2)
    records13.append({'epoch':epoch13,'train_MSE':tr13,'validation_MSE':vl13})
    if vl13<best_loss13:best_loss13=vl13;best_w13=copy.deepcopy(w13);best_epoch13=epoch13;bad13=0
    else:bad13+=1
    if bad13>=20:break
w13=best_w13.copy();assert np.isclose(np.mean((valid_x13@w13-valid_y13)**2),best_loss13)
frame13=pd.DataFrame(records13);folder13=OUT/'reports';folder13.mkdir(parents=True,exist_ok=True);frame13.to_csv(folder13/'c13_training_log.csv',index=False)
plt.plot(frame13.epoch,frame13.train_MSE,label='train');plt.plot(frame13.epoch,frame13.validation_MSE,label='validation');plt.axvline(best_epoch13,c='k',ls='--');plt.legend();plt.xlabel('epoch');plt.ylabel('MSE');savefig('c13_early_stopping')
print('Restored best epoch:',best_epoch13,'stopped:',epoch13,'max norm:',np.linalg.norm(projected13,axis=1))
# 玩具影像水平反轉保留「物體存在」標籤，不保證保留文字／數字語意
image13=np.zeros((8,8));image13[2:6,1:3]=1
assert np.fliplr(image13).shape==image13.shape

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。